# Testing proc sql is closer to integration test than others
- we are relying on some data

 I need to make

 tests/integrationtest/test_sp_isreporter

%sql
-- Expect IsReporter = 1 for user 163598

CREATE OR REPLACE TEMP VIEW test_result AS
SELECT *
FROM (
  CALL tel_unified_reporting_gold.sp_isreporter(163598)
)
WHERE IsReporter != 1;


# Here we would throw so we have a method of seeing a fail from a test where we dont have a testrunner
%py
row_count = spark.table("test_result").count()

if row_count > 0:
    raise Exception("❌ test_sp_isreporter FAILED: unexpected IsReporter value")
else:
    print("✅ test_sp_isreporter PASSED")


In [0]:
@pytest.mark.integration_sql
def test_sp_isreporter():
    df = spark.sql("""
        CALL tel_unified_reporting_gold.sp_isreporter(163598)
    """)
    result = df.collect()[0]["IsReporter"]
    assert result == 1


## 
exit_code = pytest.main([
    "integration-tests",
    "-m", "integration_sql",
    "-v",
    "-s"
])

if exit_code != 0:
    raise Exception("❌ SQL integration tests failed")
